# LLM Benchmarking in Automotive Engineering: Visualization Dashboard
This notebook runs the evaluation framework and visualizes the results from `benchmark_results.csv` using Plotly.
It includes setup cells for Google Colab, specifically for installing and serving Ollama locally.

## Step 1: Download Required Files
Because this notebook relies on custom modules we built, we need to download them into the Colab environment from the GitHub repository.

In [ ]:
!wget -q https://raw.githubusercontent.com/Adityakumar001-usn/GenAI_AEL_Task/main/evaluation_framework.py -O evaluation_framework.py
!wget -q https://raw.githubusercontent.com/Adityakumar001-usn/GenAI_AEL_Task/main/provider_adapters.py -O provider_adapters.py
!wget -q https://raw.githubusercontent.com/Adityakumar001-usn/GenAI_AEL_Task/main/metrics_engine.py -O metrics_engine.py
!wget -q https://raw.githubusercontent.com/Adityakumar001-usn/GenAI_AEL_Task/main/hallucination_detector.py -O hallucination_detector.py
!wget -q https://raw.githubusercontent.com/Adityakumar001-usn/GenAI_AEL_Task/main/prompt_dataset.csv -O prompt_dataset.csv
print("Required files downloaded successfully!")


In [ ]:
# Cell 1: Environment Setup
!pip install -q requests aiohttp tiktoken plotly nbformat google-genai

import os
import subprocess
import time
import requests

# Securely load API Keys from Google Colab Secrets (userdata)
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("API Keys loaded securely from Colab Secrets.")
except ImportError:
    print("Not running in Colab. Attempting to fall back to local environment variables.")
except userdata.SecretNotFoundError as e:
    print(f"CRITICAL: Missing API Key in Colab Secrets! {e}")
    print("Please add GEMINI_API_KEY and GROQ_API_KEY to the 🔑 Secrets tab on the left.")

# Install Ollama
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | bash

# Serve Ollama in the background
print("Starting Ollama server...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Wait for Ollama to be responsive (SME Requirement for Colab Stability)
max_retries = 15
for i in range(max_retries):
    try:
        response = requests.get("http://localhost:11434")
        if response.status_code == 200:
            print("Ollama server is up and responsive!")
            break
    except requests.exceptions.ConnectionError:
        pass
    print(f"Waiting for Ollama... ({i+1}/{max_retries})")
    time.sleep(2)

# Pull the model
print("Pulling Llama3 model (this may take a few minutes)...")
!ollama pull llama3
print("Environment setup complete.")


## E2E Diagnostic Test
Before running the full 56-prompt benchmark, we run a single 'Verification Runner' to ensure all APIs are reachable and authenticating properly.

In [ ]:
# Cell 1.5: Pre-Flight Verification Runner
import asyncio
from provider_adapters import GeminiAdapter, GroqAdapter, OllamaAdapter

async def run_diagnostics():
    print("--- E2E Connectivity Report ---")
    adapters = {
        "Gemini": GeminiAdapter(max_retries=1),
        "Groq": GroqAdapter(max_retries=1),
        "Ollama": OllamaAdapter(max_retries=1)
    }
    
    diagnostic_prompt = "Say exactly 'Hello Automotive'."
    
    for name, adapter in adapters.items():
        print(f"Testing {name}...")
        try:
            # We bypass the backoff wrapper to quickly fail if auth is wrong
            # Actually, using generate_with_retry but with max_retries=1 is fine
            res, _ = await adapter.generate_with_retry(diagnostic_prompt)
            if res:
                 print(f"  [{name}] Status: SUCCESS")
            else:
                 print(f"  [{name}] Status: FAILURE (Empty response)")
        except Exception as e:
            print(f"  [{name}] Status: FAILURE ({e})")
    
    print("-------------------------------")

await run_diagnostics()


In [ ]:
# Cell 2: Run Benchmark
# Note: Ensure evaluation_framework.py, provider_adapters.py, metrics_engine.py, hallucination_detector.py, and prompt_dataset.csv are in the working directory.
import asyncio
from evaluation_framework import EvaluationFramework

# To prevent running all 56 prompts 5 times each (which takes a long time),
# you might want to test with a smaller dataset first.
framework = EvaluationFramework()

# Execute the benchmark
await framework.run_benchmark()
print("Benchmarking finished. Results saved to benchmark_results.csv")


## Results Analysis and Visualization
The following cells load the generated data and create interactive Plotly charts.

In [ ]:
# Cell 3: Data Visualization
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

if not os.path.exists("benchmark_results.csv"):
    print("No benchmark results found. Please run the framework first.")
else:
    df = pd.read_csv("benchmark_results.csv")

    # 1. Latency by Provider
    fig1 = px.box(df, x="Provider", y="Avg_Latency_ms", color="Provider",
                  title="Average Latency Distribution by Provider")
    fig1.show()

    # 2. Tokens Per Second (TPS)
    fig2 = px.bar(df.groupby("Provider")["Avg_Tokens_Per_Second"].mean().reset_index(),
                  x="Provider", y="Avg_Tokens_Per_Second", color="Provider",
                  title="Average Tokens Per Second (TPS)")
    fig2.show()

    # 3. Consistency Score Comparison
    fig3 = px.violin(df, x="Provider", y="Consistency_Score", color="Provider", box=True,
                     title="Consistency Score Distribution across 5 Iterations")
    fig3.show()

    # 4. Hallucination Rates by Category
    hallucination_summary = df.groupby(["Category", "Provider"])["Hallucination_Flags"].sum().reset_index()
    fig4 = px.bar(hallucination_summary, x="Category", y="Hallucination_Flags", color="Provider", barmode="group",
                  title="Total Hallucination Flags by Category and Provider")
    fig4.show()

    # 5. Reasoning Quality Radar Chart
    reasoning_summary = df.groupby("Provider")["Reasoning_Score"].mean().reset_index()
    fig5 = px.line_polar(reasoning_summary, r='Reasoning_Score', theta='Provider', line_close=True,
                         title="Average Reasoning Quality Score")
    fig5.update_traces(fill='toself')
    fig5.show()
